<a href="https://colab.research.google.com/github/santarabantoosoo/semantic_song_search/blob/main/fine_tuning_pairs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aim

> In this notebook I am using the song pairs from the 1 Million dataset. However, in this trial, I am not using the score. 
I will use only songs with similarity score > 0.7. 
In a future notebook, I am planning to use the full dataset with scores. 

Based on this tutorial, Thanks to a teammate, Training Sentence Transformers with MNR Loss | Pinecone
https://www.pinecone.io/learn/fine-tune-sentence-transformers-mnr/#fast-fine-tuning


When fine-tuning with MNR loss, we will be dropping all rows with neutral or contradiction labels — keeping only the positive entailment pairs.

We will be feeding sentence A (the premise, known as the anchor) followed by sentence B (the hypothesis, when the label is 0, this is called the positive) into BERT on each step. Unlike softmax loss, we do not use the label feature.

In [ ]:
!pip install datasets

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 451 kB 28.3 MB/s 
     |████████████████████████████████| 182 kB 71.4 MB/s 
     |████████████████████████████████| 132 kB 79.0 MB/s 
     |████████████████████████████████| 212 kB 77.3 MB/s 
     |████████████████████████████████| 127 kB 66.8 MB/s 
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.24.3
    Uninstalling urllib3-1.24.3:
      Successfully uninstalled urllib3-1.24.3


## Semantic Song Search: Fine-Tune Base Model

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [58]:
%cd /content/gdrive/.shortcut-targets-by-id/1WHLBzPq6pt_F7mh3d3goIQl4MwYlTfIh/ml2_project

/content/gdrive/.shortcut-targets-by-id/1WHLBzPq6pt_F7mh3d3goIQl4MwYlTfIh/ml2_project


In [ ]:
!pip install sentence_transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 85 kB 3.3 MB/s 
     |████████████████████████████████| 5.8 MB 46.5 MB/s 
     |████████████████████████████████| 1.3 MB 48.6 MB/s 
     |████████████████████████████████| 7.6 MB 68.5 MB/s 
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125938 sha256=0a236cc5685ec166e2762c3b7b36073cf4ccc784c9feffb4ebd3c7925ebe98ee
  Stored in directory: /root/.cache/pip/wheels/5e/6f/8c/d88aec621f3f542d26fac0342bef5e693335d125f4e54aeffe
Successfully built sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer, InputExample
from tqdm.auto import tqdm  # progress bar

import pandas as pd
import numpy as np

### Data Prep

In [ ]:
from google.colab import files
files.upload()

Saving song_pairs_lyrics.csv to song_pairs_lyrics.csv


{'song_pairs_lyrics.csv': b'track1_title,track1_artist_name,track2_title,track2_artist_name,lyrics_x,lyrics_y,score\nWe Will Rock You,Queen,Walk This Way,Aerosmith,"\nBuddy, you\'re a boy, make a big noise\nPlaying in the street\nGonna be a big man someday\nYou got mud on your face\nYou big disgrace\nKicking your can\nAll over the place, singing\n\n\nWe will, we will rock you\nWe will, we will rock you\n\n\nBuddy, you\'re a young man, hard man\nShouting in the street\nGonna take on the world someday\nYou got blood on your face\nYou big disgrace\nWaving your banner\nAll over the place\n\nWe will, we will rock you (Sing it out)\nWe will, we will rock you\n\n\nBuddy, you\'re an old man, poor man\nPleading with your eyes\nGonna make you some peace someday\nYou got mud on your face\nBig disgrace\nSomebody better put you back\nInto your place\n\n\nWe will, we will rock you, sing it\nWe will, we will rock you, everybody\nWe will, we will rock you, hmm\nWe will, we will rock you, alright\n\n",

In [ ]:
pairs = pd.read_csv("song_pairs_lyrics.csv")

In [ ]:
pairs = pairs.dropna(subset = ['lyrics_x', 'lyrics_y'])

Removing duplicate values and \n

In [169]:
pairs['lyrics_x']=pairs['lyrics_x'].apply(lambda x: ', '.join(set([y.strip() for y in x.split('\n')])))
pairs['lyrics_y']=pairs['lyrics_y'].apply(lambda x: ', '.join(set([y.strip() for y in x.split('\n')])))




In [173]:
pairs['lyrics_x'] = pairs['lyrics_x'].str.replace(", ", "")
pairs['lyrics_y'] = pairs['lyrics_y'].str.replace(", ", "")

In [174]:
pairs.head()

,track1_title,track1_artist_name,track2_title,track2_artist_name,lyrics_x,lyrics_y,score
0,We Will Rock You,Queen,Walk This Way,Aerosmith,All over the placesingingBuddyyou're an old ma...,"Singing ""Heydiddle-diddle with your kitty in t...",0.100000
1,One Love,Nas,Bring Da Ruckus,Wu-Tang Clan,Mistakes happenso take heednever bust up‘Cause...,I come roughtough like an elephant tuskCreepin...,0.100000
2,Love Me,Justin Bieber,Common Denominator,Justin Bieber,Fool mefool meoh how ya do meBaby you can do n...,And I fall in love with you all over againOut ...,0.845078
3,Conteo,Don Omar,Virtual Diva,Don Omar,Sociono me ronqueporque va' a sentir el torque...,Tiene dentro esa chispa que quema transistores...,0.600000
4,Breakthru,Queen,The Great Pretender,Freddie Mercury,I get religion quickHeadlong into this ecstasy...,Pretending that you're still aroundI'm lonely ...,0.200000


In [175]:
# pairs = pairs.query('score > 0.7')

In [176]:
pairs

,track1_title,track1_artist_name,track2_title,track2_artist_name,lyrics_x,lyrics_y,score
0,We Will Rock You,Queen,Walk This Way,Aerosmith,All over the placesingingBuddyyou're an old ma...,"Singing ""Heydiddle-diddle with your kitty in t...",0.100000
1,One Love,Nas,Bring Da Ruckus,Wu-Tang Clan,Mistakes happenso take heednever bust up‘Cause...,I come roughtough like an elephant tuskCreepin...,0.100000
2,Love Me,Justin Bieber,Common Denominator,Justin Bieber,Fool mefool meoh how ya do meBaby you can do n...,And I fall in love with you all over againOut ...,0.845078
3,Conteo,Don Omar,Virtual Diva,Don Omar,Sociono me ronqueporque va' a sentir el torque...,Tiene dentro esa chispa que quema transistores...,0.600000
4,Breakthru,Queen,The Great Pretender,Freddie Mercury,I get religion quickHeadlong into this ecstasy...,Pretending that you're still aroundI'm lonely ...,0.200000
...,...,...,...,...,...,...,...
1485,Hey Mama,Kanye West,Celebration,Kanye West,Late Decemberharsh winter gave me a coldMaya A...,He looked up at meWhy is y'all lyin' for?Whaty...,0.987487
1486,The Wine Song,The Cat Empire,The Lost Song,The Cat Empire,On a river of redred wineLet's have someThe ca...,I had nine livesbut lost all of themAnd my eye...,0.933645
1487,By Your Side,Jadakiss,One Mic,Nas,I'd find youWhen it’s too hot for the hammerAn...,All I need is one lifeone tryone breathI'm one...,0.502491
1488,Never Ending Math Equation,Modest Mouse,Broke,Modest Mouse,Oh my GodI've gottagottagottagotta move onWell...,You said that I was an asshole and I'd paid th...,0.903929


In [177]:
pairs

,track1_title,track1_artist_name,track2_title,track2_artist_name,lyrics_x,lyrics_y,score
0,We Will Rock You,Queen,Walk This Way,Aerosmith,All over the placesingingBuddyyou're an old ma...,"Singing ""Heydiddle-diddle with your kitty in t...",0.100000
1,One Love,Nas,Bring Da Ruckus,Wu-Tang Clan,Mistakes happenso take heednever bust up‘Cause...,I come roughtough like an elephant tuskCreepin...,0.100000
2,Love Me,Justin Bieber,Common Denominator,Justin Bieber,Fool mefool meoh how ya do meBaby you can do n...,And I fall in love with you all over againOut ...,0.845078
3,Conteo,Don Omar,Virtual Diva,Don Omar,Sociono me ronqueporque va' a sentir el torque...,Tiene dentro esa chispa que quema transistores...,0.600000
4,Breakthru,Queen,The Great Pretender,Freddie Mercury,I get religion quickHeadlong into this ecstasy...,Pretending that you're still aroundI'm lonely ...,0.200000
...,...,...,...,...,...,...,...
1485,Hey Mama,Kanye West,Celebration,Kanye West,Late Decemberharsh winter gave me a coldMaya A...,He looked up at meWhy is y'all lyin' for?Whaty...,0.987487
1486,The Wine Song,The Cat Empire,The Lost Song,The Cat Empire,On a river of redred wineLet's have someThe ca...,I had nine livesbut lost all of themAnd my eye...,0.933645
1487,By Your Side,Jadakiss,One Mic,Nas,I'd find youWhen it’s too hot for the hammerAn...,All I need is one lifeone tryone breathI'm one...,0.502491
1488,Never Ending Math Equation,Modest Mouse,Broke,Modest Mouse,Oh my GodI've gottagottagottagotta move onWell...,You said that I was an asshole and I'd paid th...,0.903929


Since we will later use bert-base-uncasesd, I will lower case the lyrics now 

### Splitting 

In [228]:
from sklearn.model_selection import train_test_split

train_set, temp = train_test_split(pairs, test_size=0.2, random_state=1)

valid_set, test_set = train_test_split(temp, test_size=0.5, random_state=1) 

## Evaluating before fine tuning 

We first need to reformat the data using the InputExample class — passing the sentence features as texts and similarity scores to the label argument.

In [229]:
pairs.head()

,track1_title,track1_artist_name,track2_title,track2_artist_name,lyrics_x,lyrics_y,score
0,We Will Rock You,Queen,Walk This Way,Aerosmith,All over the placesingingBuddyyou're an old ma...,"Singing ""Heydiddle-diddle with your kitty in t...",0.100000
1,One Love,Nas,Bring Da Ruckus,Wu-Tang Clan,Mistakes happenso take heednever bust up‘Cause...,I come roughtough like an elephant tuskCreepin...,0.100000
2,Love Me,Justin Bieber,Common Denominator,Justin Bieber,Fool mefool meoh how ya do meBaby you can do n...,And I fall in love with you all over againOut ...,0.845078
3,Conteo,Don Omar,Virtual Diva,Don Omar,Sociono me ronqueporque va' a sentir el torque...,Tiene dentro esa chispa que quema transistores...,0.600000
4,Breakthru,Queen,The Great Pretender,Freddie Mercury,I get religion quickHeadlong into this ecstasy...,Pretending that you're still aroundI'm lonely ...,0.200000


In [261]:
from sentence_transformers import InputExample

samples = []
for index, row in valid_set.iterrows():
    samples.append(InputExample(
        texts=[row['lyrics_x'], row['lyrics_y']],
        label=row['score']
    ))


To evaluate the models, we need to initialize the appropriate evaluator object. As we are evaluating continuous similarity scores, we use the EmbeddingSimilarityEvaluator

In [262]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

evaluator = EmbeddingSimilarityEvaluator.from_input_examples(
    samples, write_csv=False
)

And with that, we’re ready to begin evaluation. We load our model as a SentenceTransformer object and pass the model to our evaluator.

The evaluator outputs the * Spearman’s rank correlation* between the cosine similarity scores calculated from the model’s output embeddings and the similarity scores provided in our dataset. A high correlation between the two values outputs a value close to *+1*, and no correlation would output *0*.

In [263]:
from sentence_transformers import models, SentenceTransformer

base_model = SentenceTransformer('models/all-MiniLM-L12-v2')
# base_model = base_model.to(torch.device('cuda')) # use GPU


base_model_performance = evaluator(base_model)

In [264]:
print(base_model_performance)

0.0910371902315363


## Training

In [265]:
from sklearn.metrics.pairwise import itertools
from sentence_transformers import InputExample
from tqdm.auto import tqdm  # so we see progress bar

train_samples = []
for index, row in train_set.iterrows():
    train_samples.append(InputExample(
        texts=[row['lyrics_x'], row['lyrics_y']]
    ))

In [266]:
from sentence_transformers import datasets

batch_size = 32

loader = datasets.NoDuplicatesDataLoader(
    train_samples, batch_size=batch_size)

In [267]:
model1 = base_model

In [268]:
from sentence_transformers import losses

loss = losses.MultipleNegativesRankingLoss(model1)

In [269]:
epochs = 3
warmup_steps = int(len(loader) * epochs * 0.1)

model1.fit(
    train_objectives=[(loader, loss)],
    epochs=epochs,
    warmup_steps=warmup_steps,
    output_path='./sbert_song_pairs_mnr2',
    show_progress_bar=True
)  # I set 'show_progress_bar=False' as it printed every step
#    on to a new line

Epoch:   0%|          | 0/3 [00:00<?, ?it/s]

Iteration:   0%|          | 0/37 [00:00<?, ?it/s]

Iteration:   0%|          | 0/37 [00:00<?, ?it/s]

Iteration:   0%|          | 0/37 [00:00<?, ?it/s]

## Evaluating 

To evaluate the models, we need to initialize the appropriate evaluator object. As we are evaluating continuous similarity scores, we use the EmbeddingSimilarityEvaluator

In [259]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

evaluator = EmbeddingSimilarityEvaluator.from_input_examples(
    samples, write_csv=False, 
)

And with that, we’re ready to begin evaluation. We load our model as a SentenceTransformer object and pass the model to our evaluator.

The evaluator outputs the * Spearman’s rank correlation* between the cosine similarity scores calculated from the model’s output embeddings and the similarity scores provided in our dataset. A high correlation between the two values outputs a value close to *+1*, and no correlation would output *0*.

In [271]:
fine_tuned_model_performance = evaluator(model1)
fine_tuned_model_performance

0.13966900417024414